# Home Energy Helper Agent

This notebook starts with a basic RAG baseline, then adds a tool-using agent that can compare two bill periods and search DOE Energy Saver guidance.

In [13]:
import json

from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
from pydantic import BaseModel, Field
from pydantic_ai import Agent
from minsearch import Index

openai_client = OpenAI()

## 1. Load Your Data

Possible knowledge sources for this project:

- Curated U.S. Department of Energy Energy Saver pages and guides (selected for this prototype): useful, practical guidance with stable source links, and quick to package as local documents.
- The U.S. Energy Information Administration (EIA) Open Data API: useful for regional electricity prices and trends; requires an API key and would add setup before the first demo.
- Synthetic monthly household bills: useful for testing charts and bill comparisons right away, but they cannot explain why a household's usage changed.

This prototype uses a local JSON file of short, topic-focused Energy Saver excerpts with source URLs. That keeps notebook runs repeatable and lets every guidance answer cite its source. The bill-comparison app should calculate kWh and cost changes separately; retrieved guidance cannot establish the cause of a usage change.

In [14]:
from pathlib import Path

data_path = Path("data/energy_saving_guides.json")
if not data_path.exists():
    data_path = Path("../data/energy_saving_guides.json")

with data_path.open(encoding="utf-8") as file:
    documents = json.load(file)

print(f"Loaded {len(documents)} Energy Saver documents from {data_path}")
print("Fields:", ", ".join(documents[0].keys()))

Loaded 5 Energy Saver documents from ..\data\energy_saving_guides.json
Fields: title, topic, content, source_name, source_url


## 2. Build the Index

minsearch builds an in-memory TF-IDF index over the fields you specify.

In [15]:
index = Index(
    text_fields=["title", "topic", "content"],
    keyword_fields=["source_name", "source_url"],
)

index.fit(documents)

## 3. Basic RAG Functions

These functions preserve the original single-search RAG baseline. The agent below reuses `search` as one of its tools and adds a focused bill-comparison tool.

In [16]:
def search(query):
    return index.search(query, num_results=5)

In [17]:
rag_instructions = """
You are a careful home energy helper. Use the retrieved U.S. Department of Energy context to answer questions about practical ways to reduce household electricity use. 
Give clear, actionable checks and distinguish general guidance from facts about this particular home.

Do not infer why a monthly kWh total changed: bills show the change, but not its cause. Do not calculate bill comparisons; 
the app handles those calculations. If the cause or a tailored recommendation depends on missing details, say what cannot be determined and ask a short, 
relevant follow-up question (for example, whether the billing period was longer or whether heating, a space heater, or air conditioning was used more).

Use only claims supported by the supplied context. Include a Markdown link to the source URL for every guidance-based answer, using the source title as the link text.
If the context does not support an answer, say so and ask what information would help; do not invent a recommendation or source.
""".strip()


def build_prompt(query, search_results):
    search_result_json = json.dumps(search_results, indent=2)

    user_prompt = f"""
<QUESTION>
{query}
</QUESTION>

<CONTEXT>
{search_result_json}
</CONTEXT>
""".strip()

    return user_prompt

In [18]:
def llm(user_prompt, instructions=None, model='gpt-4o-mini'):
    messages = []

    if instructions is not None:
        messages.append({
            "role": "system",
            "content": instructions
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return response.output_text

In [19]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt, rag_instructions)
    return answer

## 4. PydanticAI Tools and Agent

`HomeEnergyTools` follows the course's class-based tools pattern. PydanticAI receives its bound methods through `Agent(tools=[...])` and manages the model/tool loop; the methods print their own calls so you can see what was used.

The original `search`, `build_prompt`, `llm`, and `rag` functions remain available as the RAG baseline. The PydanticAI agent below reuses `search` through `search_energy_guidance` and adds `compare_monthly_bills`.

In [20]:
from math import isfinite


class BillPeriod(BaseModel):
    """Bill details for one month; provide billing_days when known."""

    month: str = Field(description="The bill-period label exactly as the user gave it; do not infer a year.")
    kwh: float = Field(ge=0, description="Electricity use in kilowatt-hours.")
    bill_amount: float = Field(ge=0, description="Total bill amount in the currency stated by the user.")
    billing_days: int | None = Field(default=None, ge=1, description="Number of billing days, or null if unknown.")


class HomeEnergyTools:
    """Small, focused tools registered with the PydanticAI agent."""

    def search_energy_guidance(self, question: str) -> list[dict]:
        """Search local DOE Energy Saver records and return relevant excerpts with source links."""
        print(f"TOOL CALL: search_energy_guidance({question!r})")
        results = search(question)
        print(f"TOOL RESULT: search_energy_guidance returned {len(results)} records")
        return [
            {
                "title": result.get("title"),
                "topic": result.get("topic"),
                "content": result.get("content"),
                "source_name": result.get("source_name"),
                "source_url": result.get("source_url"),
            }
            for result in results
        ]

    @staticmethod
    def _validate_period(period: dict, label: str) -> None:
        if not isinstance(period.get("month"), str) or not period["month"].strip():
            raise ValueError(f"{label}.month must be a non-empty string")

        for field in ("kwh", "bill_amount"):
            value = period.get(field)
            if isinstance(value, bool) or not isinstance(value, (int, float)):
                raise ValueError(f"{label}.{field} must be a number")
            if not isfinite(value) or value < 0:
                raise ValueError(f"{label}.{field} must be finite and non-negative")

        days = period.get("billing_days")
        if days is not None and (isinstance(days, bool) or not isinstance(days, int) or days < 1):
            raise ValueError(f"{label}.billing_days must be a positive integer or null")

    def compare_monthly_bills(
        self,
        previous_period: BillPeriod,
        current_period: BillPeriod,
    ) -> dict:
        """Compare two user-provided bill periods; report totals and daily averages when possible."""
        previous_period = previous_period.model_dump()
        current_period = current_period.model_dump()
        self._validate_period(previous_period, "previous_period")
        self._validate_period(current_period, "current_period")
        print(
            "TOOL CALL: compare_monthly_bills "
            f"({previous_period['month']} -> {current_period['month']})"
        )

        def change(current, previous):
            absolute = current - previous
            percent = None if previous == 0 else (absolute / previous) * 100
            return round(absolute, 2), None if percent is None else round(percent, 1)

        kwh_change, kwh_change_percent = change(current_period["kwh"], previous_period["kwh"])
        bill_change, bill_change_percent = change(current_period["bill_amount"], previous_period["bill_amount"])

        previous_days = previous_period["billing_days"]
        current_days = current_period["billing_days"]
        if previous_days is None or current_days is None:
            previous_kwh_per_day = current_kwh_per_day = daily_kwh_change = None
            previous_bill_per_day = current_bill_per_day = daily_bill_change = None
        else:
            previous_kwh_per_day = previous_period["kwh"] / previous_days
            current_kwh_per_day = current_period["kwh"] / current_days
            previous_bill_per_day = previous_period["bill_amount"] / previous_days
            current_bill_per_day = current_period["bill_amount"] / current_days
            daily_kwh_change = current_kwh_per_day - previous_kwh_per_day
            daily_bill_change = current_bill_per_day - previous_bill_per_day
            previous_kwh_per_day = round(previous_kwh_per_day, 2)
            current_kwh_per_day = round(current_kwh_per_day, 2)
            previous_bill_per_day = round(previous_bill_per_day, 2)
            current_bill_per_day = round(current_bill_per_day, 2)
            daily_kwh_change = round(daily_kwh_change, 2)
            daily_bill_change = round(daily_bill_change, 2)

        result = {
            "previous_month": previous_period["month"],
            "current_month": current_period["month"],
            "usage_change_kwh": kwh_change,
            "usage_change_percent": kwh_change_percent,
            "bill_change_amount": bill_change,
            "bill_change_percent": bill_change_percent,
            "previous_kwh_per_day": previous_kwh_per_day,
            "current_kwh_per_day": current_kwh_per_day,
            "daily_usage_change_kwh": daily_kwh_change,
            "previous_bill_per_day": previous_bill_per_day,
            "current_bill_per_day": current_bill_per_day,
            "daily_bill_change": daily_bill_change,
            "note": "This comparison shows what changed; it does not identify the cause. A null percent means the previous value was zero. Daily averages are null when either billing period length is missing.",
        }
        print("TOOL RESULT: compare_monthly_bills completed")
        return result


home_energy_tools = HomeEnergyTools()

In [21]:
agent_instructions = """
You are the Home Energy Helper, an assistant for understanding household electricity bills and finding practical ways to save energy.

Use your tools:
- For a month-over-month bill question, call compare_monthly_bills only when the user supplied both months' kWh and bill amounts. Preserve month labels and numeric values exactly; never infer a year. Pass null for unknown billing days. If required values are missing, ask only for those values.
- For questions about energy-saving, heating, appliance efficiency, or home energy checks, call search_energy_guidance. Use the retrieved excerpts and their source URLs; link every guidance-based answer to relevant DOE sources.
- You may call both tools when the user asks why a bill changed and what to check.

Be accurate:
- State bill differences as observations, never as causes. The comparison cannot prove that weather, heating, rates, billing days, or an appliance caused a change.
- For high-bill questions, give the computed changes first, then up to three supported checks, and finish with one targeted follow-up about the most useful missing detail.
- Use only guidance explicitly supported by retrieved excerpts. Do not invent savings figures, costs, appliance performance, rates, current weather, rebates, or source links.
- This version has no live weather, electricity/gas rate, appliance-use, or rebate lookup. Do not claim which heating option is cheapest or calculate appliance payback.
- Keep answers concise and practical. Separate bill arithmetic from general DOE guidance.
""".strip()

home_energy_agent = Agent(
    "openai:gpt-4o-mini",
    name="home_energy_helper",
    instructions=agent_instructions,
    tools=[
        home_energy_tools.search_energy_guidance,
        home_energy_tools.compare_monthly_bills,
    ],
)

In [22]:
async def run_agent(user_question: str) -> str:
    """Run PydanticAI asynchronously; the framework manages tool-call iterations."""
    result = await home_energy_agent.run(user_question)
    return result.output


## 5. Try the Agent

The figures below are an illustrative example to demonstrate both tools. In a real app run, pass the user's actual bill details.

In [23]:
answer = await run_agent(
    "For this illustrative example, my electricity use went from 620 kWh and $124 over 31 days in January to 790 kWh and $158 over 28 days in February. Why is my bill higher, and what should I check?"
)
print(answer)


TOOL CALL: compare_monthly_bills (January -> February)
TOOL RESULT: compare_monthly_bills completed
Your electricity bill increased due to several factors:

1. **Usage Changes**: 
   - Your usage went up by **170 kWh** (from 620 kWh to 790 kWh), which is a **27.4% increase**.
   - Daily average usage rose from **20 kWh/day** in January to **28.21 kWh/day** in February.

2. **Bill Changes**: 
   - Your total bill increased by **$34** (from $124 to $158), which is also a **27.4% increase**.
   - Daily costs increased from **$4/day** to **$5.64/day**, a change of **$1.64** per day.

### Suggested Checks
Here are some areas to investigate that might be contributing to the increase:

1. **Appliance Use**: Review the use of major appliances, especially those that consume significant energy like heaters or air conditioning, as their usage can spike in different seasons.
   
2. **Weather**: Consider any weather changes that could have led to increased heating or cooling needs. 

3. **Electrica